# Spontaneous Raman Spectra Pipeline Template

below listed out all the helper functions wrote

1. from helpers import save_and_load
    1. save_spectrum(fileType: 'txt' or 'csv', outputpath as Path variable, wavenumber as nparray, intensity as nparray, [optional] background curve as nparray, header defaults to False means no header) -> None:
        saves the wavenumber and intensity data to a file either csv or txt format
    2. read_spectrum(path as Path variable, header defaults to False) -> Tuple[np.ndarry]
        read a spectrum file, the first coln will be wavenumber, second coln will be intensity, [optional] third coln will be background
    3. save_ratios(path as Path variable, data as a List of Dict[str, any], header defaults to true) -> None:
        saves a list of dictionaries containing ratio data to a csv file
    4. load_ratios(path as Path variable) -> pd.DataFrame:
        loads ratio data from a csv or a txt file into a pandas DataFrame
2. from helpers import preprocessing
    1. fitted_bg_subtraction(wavenumber as nparray, intensity as nparray, range start defaults to 1000, range end defaults to 3900) -> Tuple[nparray, nparray, nparry]:
        performs a background subtraction by fitting a Gaussian curve to the wavenumber range from (range_start to range_end), saves the fitted Gaussia curve to the third nparray. This function only fits to one Gaussian peak
    2. spectra_bg_subtraction(wavenumber as nparray, intensity as nparray, background spectra wavenumber as nparray, background spectra intensity sa nparray) -> Tuple[nparray, nparray]:
        performs a background subtraction if you provide a background spectra, no background fitting
    3. simple_baseline_correction(wavenumber as nparray, intensity as nparray, cell_silent_start as int, cell_silent_end as int, trim defaults to None, min_max defaults to True) -> Tuple[nparray, nparray]:
        performs a simple baseline correction on a Raman spectrum by setting the average intensity value from 2300 to 2600 as zero, everything below will be set to zero. min_max True if perform a min-max normalization
    4. polynomial_baseline_correction(wavenumber as nparray, intensity as nparray) -> Tuple[nparray, nparray]:
        performs a baseline correction by fitting a polynomial to the spectrum
    5. smoothing(wavenumber as nparray, intensity as nparray, window_length as int defaults to 11, polyorder as int defaults to 2) -> Tuple[nparray, nparray]:
        performs a savitzky-golay filter for a spectrum to smooth
3. from helpers import analysis
    1. peak_ratio_calculator(wavenumber as nparray, intensity as nparray, peak 1 location, peak 2 location, [optional] background intensity) -> float:
        calculate a ratio between the intensity at two peak wavenumber location. If a background intensity is provided, peak 2 will be seen as a water background
    2. area_under_curve_ratio_calculator(wavenumber as nparry, intensity as nparray, peak range 1, peak range 2, [optional] background intensity) -> float:
        calculate the area underr the curve for a specified range of wavenumber in a Raman spectrum. If a background intensity is provided, peak 2's intensity will be read from background intensity
    3. t_test_calculator(group_1 as nparray, group_2 as nparray) -> tuple:
        performs a t-test between two groups of data
4. from helpers import plots
    
    nparray is able to used by matplotlib, for all this part, please see the script yourself since might be a bit complicate to explain here.

### Library Loading

In [ ]:
import os, sys
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import pandas as pd

os.chdir(os.getcwd())

from helpers import save_and_load as sl
from helpers import preprocessing as prep
from helpers import analysis
from helpers import plots
import pathlib as Path

Load file path to a df table

In [14]:
base_dir = r'D:\Chrome\Workspace\Projects\Rapamycin\data\spontaneous\Rapamycin_2'
base_dir = Path.Path(base_dir)
all_files = sorted(base_dir.glob("*/*/*/*.txt"))
rows = []

for path in all_files:
    name_lower = path.name.lower()
    if ("fitted" in name_lower or "baseline" in name_lower or "bg" in name_lower or "background" in name_lower):
        continue
    
    if ("ctrl" not in name_lower and "rapa" not in name_lower):
        continue

    bg_path = path.with_name(path.stem + '_BG.txt')
    if not bg_path.exists():
        print(f"Warning: Background file {bg_path} does not exist for {path}")
        continue

    rel = path.relative_to(base_dir)
    day, organ, group, filename = rel.parts
    
    rows.append({
        'day': day.split('_')[1],
        'organ': organ,
        'group': group,
        'filename': path.stem,
        'spectrum_path': path,
        'background_path': bg_path,
        'has_background': bg_path.exists()
    })

file_table = pd.DataFrame(rows)

with open('file_table.csv', 'w') as f:
    file_table.to_csv(f, index=False)
file_table.head(20)

,day,organ,group,filename,spectrum_path,background_path,has_background
0,D10,Fatbody,1_RAPA,D10_RAPA_1,D:\Chrome\Workspace\Projects\Rapamycin\data\sp...,D:\Chrome\Workspace\Projects\Rapamycin\data\sp...,True
1,D10,Fatbody,1_RAPA,D10_RAPA_2,D:\Chrome\Workspace\Projects\Rapamycin\data\sp...,D:\Chrome\Workspace\Projects\Rapamycin\data\sp...,True
2,D10,Fatbody,1_RAPA,D10_RAPA_3,D:\Chrome\Workspace\Projects\Rapamycin\data\sp...,D:\Chrome\Workspace\Projects\Rapamycin\data\sp...,True
3,D10,Fatbody,1_RAPA,D10_RAPA_4,D:\Chrome\Workspace\Projects\Rapamycin\data\sp...,D:\Chrome\Workspace\Projects\Rapamycin\data\sp...,True
4,D10,Fatbody,1_RAPA,D10_RAPA_5,D:\Chrome\Workspace\Projects\Rapamycin\data\sp...,D:\Chrome\Workspace\Projects\Rapamycin\data\sp...,True
5,D10,Fatbody,2_RAPA,D10_RAPA_1,D:\Chrome\Workspace\Projects\Rapamycin\data\sp...,D:\Chrome\Workspace\Projects\Rapamycin\data\sp...,True
6,D10,Fatbody,2_RAPA,D10_RAPA_2,D:\Chrome\Workspace\Projects\Rapamycin\data\sp...,D:\Chrome\Workspace\Projects\Rapamycin\data\sp...,True
7,D10,Fatbody,2_RAPA,D10_RAPA_3,D:\Chrome\Workspace\Projects\Rapamycin\data\sp...,D:\Chrome\Workspace\Projects\Rapamycin\data\sp...,True
8,D10,Fatbody,2_RAPA,D10_RAPA_4,D:\Chrome\Workspace\Projects\Rapamycin\data\sp...,D:\Chrome\Workspace\Projects\Rapamycin\data\sp...,True
9,D10,Fatbody,2_RAPA,D10_RAPA_5,D:\Chrome\Workspace\Projects\Rapamycin\data\sp...,D:\Chrome\Workspace\Projects\Rapamycin\data\sp...,True


## Analysis Plan
1. sum all the intensity in a group together
2. take the average
3. fit the background
4. overlay rapa vs ctrl
5. plot, zoom-in plot

### Gut Analysis

In [ ]:
# D10 Gut Analysis
d10_gut_1_rapa_files = file_table[
    (file_table['day'] == 'D10') &
    (file_table['organ'] == 'Gut') &
    (file_table['group'] == '1_RAPA')
]
d10_gut_2_rapa_files = file_table[
    (file_table['day'] == 'D10') &
    (file_table['organ'] == 'Gut') &
    (file_table['group'] == '2_RAPA')
]
d10_gut_4_ctrl_files = file_table[
    (file_table['day'] == 'D10') &
    (file_table['organ'] == 'Gut') &
    (file_table['group'] == '4_CTRL')
]
d10_gut_5_ctrl_files = file_table[
    (file_table['day'] == 'D10') &
    (file_table['organ'] == 'Gut') &
    (file_table['group'] == '5_CTRL')
]


# D40 Gut Analysis
d40_gut_1_rapa_files = file_table[
    (file_table['day'] == 'D40') &
    (file_table['organ'] == 'Gut') &
    (file_table['group'] == '1_RAPA')
]
d40_gut_2_rapa_files = file_table[
    (file_table['day'] == 'D40') &
    (file_table['organ'] == 'Gut') &
    (file_table['group'] == '2_RAPA')
]
d40_gut_4_ctrl_files = file_table[
    (file_table['day'] == 'D40') &
    (file_table['organ'] == 'Gut') &
    (file_table['group'] == '4_CTRL')
]
d40_gut_5_ctrl_files = file_table[
    (file_table['day'] == 'D40') &
    (file_table['organ'] == 'Gut') &
    (file_table['group'] == '5_CTRL')
]

### Ovary Analysis

In [ ]:
d10_ovary_1_rapa_files = file_table[
    (file_table['day'] == 'D10') &
    (file_table['organ'] == 'Ovary') &
    (file_table['group'] == '1_RAPA')
]
d10_ovary_2_rapa_files = file_table[
    (file_table['day'] == 'D10') &
    (file_table['organ'] == 'Ovary') &
    (file_table['group'] == '2_RAPA')
]
d10_ovary_4_ctrl_files = file_table[
    (file_table['day'] == 'D10') &
    (file_table['organ'] == 'Ovary') &
    (file_table['group'] == '4_CTRL')
]
d10_ovary_5_ctrl_files = file_table[
    (file_table['day'] == 'D10') &
    (file_table['organ'] == 'Ovary') &
    (file_table['group'] == '5_CTRL')
]
d40_ovary_1_rapa_files = file_table[
    (file_table['day'] == 'D40') &
    (file_table['organ'] == 'Ovary') &
    (file_table['group'] == '1_RAPA')
]
d40_ovary_2_rapa_files = file_table[
    (file_table['day'] == 'D40') &
    (file_table['organ'] == 'Ovary') &
    (file_table['group'] == '2_RAPA')
]
d40_ovary_4_ctrl_files = file_table[
    (file_table['day'] == 'D40') &
    (file_table['organ'] == 'Ovary') &
    (file_table['group'] == '4_CTRL')
]
d40_ovary_5_ctrl_files = file_table[
    (file_table['day'] == 'D40') &
    (file_table['organ'] == 'Ovary') &
    (file_table['group'] == '5_CTRL')
]

### Fatbody Analysis

In [ ]:
d10_fb_1_rapa_files = file_table[
    (file_table['day'] == 'D10') &
    (file_table['organ'] == 'Fatbody') &
    (file_table['group'] == '1_RAPA')
]
d10_fb_2_rapa_files = file_table[
    (file_table['day'] == 'D10') &
    (file_table['organ'] == 'Fatbody') &
    (file_table['group'] == '2_RAPA')
]
d10_fb_4_ctrl_files = file_table[
    (file_table['day'] == 'D10') &
    (file_table['organ'] == 'Fatbody') &
    (file_table['group'] == '4_CTRL')
]
d10_fb_5_ctrl_files = file_table[
    (file_table['day'] == 'D10') &
    (file_table['organ'] == 'Fatbody') &
    (file_table['group'] == '5_CTRL')
]
d40_fb_1_rapa_files = file_table[
    (file_table['day'] == 'D40') &
    (file_table['organ'] == 'Fatbody') &
    (file_table['group'] == '1_RAPA')
]
d40_fb_2_rapa_files = file_table[
    (file_table['day'] == 'D40') &
    (file_table['organ'] == 'Fatbody') &
    (file_table['group'] == '2_RAPA')
]
d40_fb_4_ctrl_files = file_table[
    (file_table['day'] == 'D40') &
    (file_table['organ'] == 'Fatbody') &
    (file_table['group'] == '4_CTRL')
]
d40_fb_5_ctrl_files = file_table[
    (file_table['day'] == 'D40') &
    (file_table['organ'] == 'Fatbody') &
    (file_table['group'] == '5_CTRL')
]